In [1]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.model_selection import train_test_split
from sklearn.preprocessing import OneHotEncoder
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline

from sklearn.linear_model import LogisticRegression
from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import RandomForestClassifier

from sklearn.metrics import(
    accuracy_score,
    classification_report,
    confusion_matrix
)

import warnings
warnings.filterwarnings("ignore")

In [2]:
df=pd.read_csv("../data/Dataset.csv")

In [3]:
df.head()

,show_id,type,title,director,country,date_added,release_year,rating,duration,listed_in
0,s1,Movie,Dick Johnson Is Dead,Kirsten Johnson,United States,9/25/2021,2020,PG-13,90 min,Documentaries
1,s3,TV Show,Ganglands,Julien Leclercq,France,9/24/2021,2021,TV-MA,1 Season,"Crime TV Shows, International TV Shows, TV Act..."
2,s6,TV Show,Midnight Mass,Mike Flanagan,United States,9/24/2021,2021,TV-MA,1 Season,"TV Dramas, TV Horror, TV Mysteries"
3,s14,Movie,Confessions of an Invisible Girl,Bruno Garotti,Brazil,9/22/2021,2021,TV-PG,91 min,"Children & Family Movies, Comedies"
4,s8,Movie,Sankofa,Haile Gerima,United States,9/24/2021,1993,TV-MA,125 min,"Dramas, Independent Movies, International Movies"


In [4]:
df.shape


(8790, 10)

In [5]:
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 8790 entries, 0 to 8789
Data columns (total 10 columns):
 #   Column        Non-Null Count  Dtype 
---  ------        --------------  ----- 
 0   show_id       8790 non-null   object
 1   type          8790 non-null   object
 2   title         8790 non-null   object
 3   director      8790 non-null   object
 4   country       8790 non-null   object
 5   date_added    8790 non-null   object
 6   release_year  8790 non-null   int64 
 7   rating        8790 non-null   object
 8   duration      8790 non-null   object
 9   listed_in     8790 non-null   object
dtypes: int64(1), object(9)
memory usage: 686.8+ KB


In [6]:
df["type"].value_counts()

type
Movie      6126
TV Show    2664
Name: count, dtype: int64

In [7]:
df["type"].value_counts(normalize=True)*100

type
Movie      69.692833
TV Show    30.307167
Name: proportion, dtype: float64

In [8]:
df["rating"].value_counts()

rating
TV-MA       3205
TV-14       2157
TV-PG        861
R            799
PG-13        490
TV-Y7        333
TV-Y         306
PG           287
TV-G         220
NR            79
G             41
TV-Y7-FV       6
NC-17          3
UR             3
Name: count, dtype: int64

In [9]:
df["duration"].value_counts().head(15)

duration
1 Season     1791
2 Seasons     421
3 Seasons     198
90 min        152
97 min        146
93 min        146
94 min        146
91 min        144
95 min        137
96 min        130
92 min        129
102 min       122
98 min        120
99 min        118
101 min       116
Name: count, dtype: int64

In [10]:
for col in df.columns:
    print(f"{col}:{df[col].nunique()} unique value")

show_id:8790 unique value
type:2 unique value
title:8787 unique value
director:4528 unique value
country:86 unique value
date_added:1713 unique value
release_year:74 unique value
rating:14 unique value
duration:220 unique value
listed_in:513 unique value


In [11]:
df["listed_in"].value_counts()

listed_in
Dramas, International Movies                                362
Documentaries                                               359
Stand-Up Comedy                                             334
Comedies, Dramas, International Movies                      274
Dramas, Independent Movies, International Movies            252
                                                           ... 
Anime Features                                                1
Action & Adventure, Horror Movies, Independent Movies         1
Action & Adventure, Classic Movies, International Movies      1
Cult Movies, Independent Movies, Thrillers                    1
Classic & Cult TV, Crime TV Shows, TV Dramas                  1
Name: count, Length: 513, dtype: int64

In [12]:
df["country"].value_counts()

country
United States     3240
India             1057
United Kingdom     638
Pakistan           421
Not Given          287
                  ... 
Iran                 1
West Germany         1
Greece               1
Zimbabwe             1
Soviet Union         1
Name: count, Length: 86, dtype: int64

In [13]:
X=df[["country","release_year","rating","listed_in"]]

y=df["type"]

In [14]:
print("Feature shape",X.shape)
print("Target shape",y.shape)

Feature shape (8790, 4)
Target shape (8790,)


In [15]:
from sklearn.model_selection import train_test_split

X_train,X_test,y_train,y_test=train_test_split(
    X,
    y,
    test_size=0.2,  #20% testing, 80% training
    random_state=42, #Makes the split reproducible
    stratify=y
)

In [16]:
print("X_train:", X_train.shape)
print("X_test:", X_test.shape)

print("\ny_train:", y_train.shape)
print("y_test:", y_test.shape)

X_train: (7032, 4)
X_test: (1758, 4)

y_train: (7032,)
y_test: (1758,)


In [17]:
print("Original distribution")
print(y.value_counts(normalize=True)*100)

print("Training distribution")
print(y_train.value_counts(normalize=True)*100)

print("Testing distribution")
print(y_test.value_counts(normalize=True)*100)


Original distribution
type
Movie      69.692833
TV Show    30.307167
Name: proportion, dtype: float64
Training distribution
type
Movie      69.695677
TV Show    30.304323
Name: proportion, dtype: float64
Testing distribution
type
Movie      69.681456
TV Show    30.318544
Name: proportion, dtype: float64


In [18]:
X_train.dtypes

country         object
release_year     int64
rating          object
listed_in       object
dtype: object

In [19]:
X_train.head(10)

,country,release_year,rating,listed_in
5268,India,1972,TV-14,"International Movies, Romantic Movies"
123,Pakistan,2021,TV-MA,"International TV Shows, Romantic TV Shows, TV ..."
2901,Spain,2019,TV-MA,"Dramas, International Movies"
6170,United States,2009,R,"Comedies, Dramas"
5423,India,2006,TV-14,"Comedies, Dramas, International Movies"
8590,United States,2015,TV-MA,"TV Comedies, TV Dramas"
6133,United States,2013,R,"Dramas, Thrillers"
6481,Peru,2016,TV-14,"Documentaries, International Movies"
7843,United States,2018,TV-MA,"Documentaries, Music & Musicals"
7754,Japan,2019,TV-MA,"International TV Shows, Romantic TV Shows, TV ..."


In [20]:
from sklearn.preprocessing import OneHotEncoder,MultiLabelBinarizer
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline

In [21]:
genres=[]

for item in X_train["listed_in"]:
    genre_list=[]
    for genre in item.split(","):
        genre_list.append(genre.strip())
    genres.append(genre_list)
genres[:5]

[['International Movies', 'Romantic Movies'],
 ['International TV Shows', 'Romantic TV Shows', 'TV Comedies'],
 ['Dramas', 'International Movies'],
 ['Comedies', 'Dramas'],
 ['Comedies', 'Dramas', 'International Movies']]

In [22]:
from sklearn.preprocessing import MultiLabelBinarizer
mlb=MultiLabelBinarizer()

genre_encoded=mlb.fit_transform(genres)
print("Encoded shape",genre_encoded.shape)
print("Genre_name",mlb.classes_[:10])

Encoded shape (7032, 42)
Genre_name ['Action & Adventure' 'Anime Features' 'Anime Series' 'British TV Shows'
 'Children & Family Movies' 'Classic & Cult TV' 'Classic Movies'
 'Comedies' 'Crime TV Shows' 'Cult Movies']


In [23]:
genre_df = pd.DataFrame(
    genre_encoded,
    columns=mlb.classes_,
    index=X_train.index
)

genre_df.head()

,Action & Adventure,Anime Features,Anime Series,British TV Shows,Children & Family Movies,Classic & Cult TV,Classic Movies,Comedies,Crime TV Shows,Cult Movies,...,TV Action & Adventure,TV Comedies,TV Dramas,TV Horror,TV Mysteries,TV Sci-Fi & Fantasy,TV Shows,TV Thrillers,Teen TV Shows,Thrillers
5268,0,0,0,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0
123,0,0,0,0,0,0,0,0,0,0,...,0,1,0,0,0,0,0,0,0,0
2901,0,0,0,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0
6170,0,0,0,0,0,0,0,1,0,0,...,0,0,0,0,0,0,0,0,0,0
5423,0,0,0,0,0,0,0,1,0,0,...,0,0,0,0,0,0,0,0,0,0


In [24]:
from sklearn.preprocessing import OneHotEncoder
encoder=OneHotEncoder(
    handle_unknown="ignore",
    sparse_output=False
)

In [25]:
categorical_train=encoder.fit_transform(
    X_train[["country","rating"]]
)

In [26]:
print("Encoded shape",categorical_train.shape)
print("Features names")

print(encoder.get_feature_names_out())

Encoded shape (7032, 93)
Features names
['country_Argentina' 'country_Australia' 'country_Austria'
 'country_Bangladesh' 'country_Belgium' 'country_Brazil'
 'country_Bulgaria' 'country_Cambodia' 'country_Cameroon' 'country_Canada'
 'country_Chile' 'country_China' 'country_Colombia' 'country_Croatia'
 'country_Cyprus' 'country_Czech Republic' 'country_Denmark'
 'country_Egypt' 'country_Finland' 'country_France' 'country_Germany'
 'country_Ghana' 'country_Greece' 'country_Guatemala' 'country_Hong Kong'
 'country_Hungary' 'country_Iceland' 'country_India' 'country_Indonesia'
 'country_Ireland' 'country_Israel' 'country_Italy' 'country_Jamaica'
 'country_Japan' 'country_Jordan' 'country_Kenya' 'country_Kuwait'
 'country_Lebanon' 'country_Luxembourg' 'country_Malaysia'
 'country_Mauritius' 'country_Mexico' 'country_Mozambique'
 'country_Netherlands' 'country_New Zealand' 'country_Nigeria'
 'country_Norway' 'country_Not Given' 'country_Pakistan'
 'country_Paraguay' 'country_Peru' 'country_Ph

In [27]:
categorical_df=pd.DataFrame(
    categorical_train,
    columns=encoder.get_feature_names_out(),
    index=X_train.index 
)
categorical_df.head()

,country_Argentina,country_Australia,country_Austria,country_Bangladesh,country_Belgium,country_Brazil,country_Bulgaria,country_Cambodia,country_Cameroon,country_Canada,...,rating_PG-13,rating_R,rating_TV-14,rating_TV-G,rating_TV-MA,rating_TV-PG,rating_TV-Y,rating_TV-Y7,rating_TV-Y7-FV,rating_UR
5268,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,1.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
123,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,1.0,0.0,0.0,0.0,0.0,0.0
2901,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,1.0,0.0,0.0,0.0,0.0,0.0
6170,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,1.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
5423,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,1.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0


In [28]:
year_df=X_train[["release_year"]].copy()
year_df.head(10)

,release_year
5268,1972
123,2021
2901,2019
6170,2009
5423,2006
8590,2015
6133,2013
6481,2016
7843,2018
7754,2019


In [29]:
X_train_processed = pd.concat(
    [year_df, categorical_df, genre_df],
    axis=1
)

X_train_processed.head()

,release_year,country_Argentina,country_Australia,country_Austria,country_Bangladesh,country_Belgium,country_Brazil,country_Bulgaria,country_Cambodia,country_Cameroon,...,TV Action & Adventure,TV Comedies,TV Dramas,TV Horror,TV Mysteries,TV Sci-Fi & Fantasy,TV Shows,TV Thrillers,Teen TV Shows,Thrillers
5268,1972,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0,0,0,0,0,0,0,0,0,0
123,2021,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0,1,0,0,0,0,0,0,0,0
2901,2019,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0,0,0,0,0,0,0,0,0,0
6170,2009,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0,0,0,0,0,0,0,0,0,0
5423,2006,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0,0,0,0,0,0,0,0,0,0


In [30]:
print("Original training data:", X_train.shape)
print("Processed training data:", X_train_processed.shape)

Original training data: (7032, 4)
Processed training data: (7032, 136)


In [31]:
genres_test=[]
for item in X_test["listed_in"]:
    genre_list=[]
    for genre in item.split(","):
        genre_list.append(genre.strip())

    genres_test.append(genre_list)

genres_test[:5]

[['Documentaries', 'International Movies'],
 ['Children & Family Movies'],
 ['Dramas', 'Independent Movies', 'International Movies'],
 ['Movies'],
 ['TV Comedies', 'TV Dramas']]